# 03 — LoRA adaptation

**Purpose:** Train adapter-only versions of the same Qwen checkpoint; save logs, trained-parameter count, dataset hashes and adapter files.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


### Training choices

Default: `10 examples/class`, rank 8, seed 42, two epochs. After the pilot works, add `25,50` and seed repeats. All training uses only the training split. Validation is available for selection but the shared training function does not read the test set.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU on Kaggle/Colab. CPU fine-tuning is intentionally blocked here.'
TASK='sib200'; LANG='ur'
SIZES=[10]  # after pilot, [10,25,50]
RANKS=[8]  # after pilot, [8,16] at a fixed training size
SEEDS=[42] # after pilot, [42,123,456]
MODEL_OVERRIDE = None  # set to saved English anchor path for controlled forgetting
DRY_RUN = False
print('GPU:',torch.cuda.get_device_name(0))
for seed in SEEDS:
    for n in SIZES:
        for rank in RANKS:
            run_name = f'{TASK}_lora_{LANG}_n{n}_s{seed}_r{rank}' + ('_anchor' if MODEL_OVERRIDE else '')
            if (ROOT/'outputs/checkpoints'/run_name/'complete.json').exists():
                print('Already complete; skip:',run_name); continue
            if DRY_RUN: print('Would run:',run_name); continue
            r=train_run('lora',TASK,n,seed,run_name=run_name,lang=LANG,rank=rank,model_override=MODEL_OVERRIDE)
            print('FINISHED',r['run_id'],'minutes:',round(r['wall_seconds']/60,2),
                  'peak GiB:',r['peak_gpu_allocated_gib'],'trainable %:',r['trainable_pct'])

The checkpoint contains **LoRA adapters**, not a standalone copy of the full Qwen model. For inference the base model and adapter must both be loaded. Do not combine adapters from different base checkpoints.